In [ ]:
%env GENAI_API_KEY=...
%env OPENAI_API_KEY=...
%env UFAL_API_KEY=...
%env GROQ_API_KEY=...

In [45]:
# Requires env variables
# GENAI_API_KEY=...
# OPENAI_API_KEY=...
# UFAL_API_KEY=...
# GROQ_API_KEY=...

import os
import json
import requests
from groq import Groq
from openai import OpenAI
from google import genai



class LLMClient:
    def chat(self, messages: list[dict]) -> str:
        raise NotImplementedError


class OpenAIClient(LLMClient):
    def __init__(
        self,
        model="gpt-5-mini",
        api_key=None,
        temperature=None,
        reasoning=None,
    ):
        self.client = OpenAI(api_key=api_key or os.environ["OPENAI_API_KEY"])
        self.model = model
        self.temperature = temperature
        self.reasoning = reasoning

    def _to_responses_input(self, messages):
        items = []
        for m in messages:
            role = m["role"]
            content = m["content"]

            # Responses API expects structured content items
            content_type = "output_text" if role == "assistant" else "input_text"
            items.append({
                "role": role,
                "content": [
                    {
                        "type": content_type,
                        "text": content,
                    }
                ],
            })
        return items

    def chat(self, messages):
        payload = {
            "model": self.model,
            "input": self._to_responses_input(messages),
        }

        if self.temperature is not None:
            payload["temperature"] = self.temperature

        if self.reasoning is not None:
            payload["reasoning"] = {"effort": self.reasoning}

        resp = self.client.responses.create(**payload)
        return resp.output_text


class GroqClient(LLMClient):
    def __init__(
        self,
        model="openai/gpt-oss-120b",
        api_key=None,
        temperature=None,
        reasoning=None,
    ):
        self.client = Groq(api_key=api_key or os.environ["GROQ_API_KEY"])
        self.model = model
        self.temperature = temperature
        self.reasoning = reasoning

    def chat(self, messages):
        payload = {
            "model": self.model,
            "messages": messages,
        }

        if self.temperature is not None:
            payload["temperature"] = self.temperature

        if self.reasoning is not None:
            payload["reasoning_effort"] = self.reasoning

        resp = self.client.chat.completions.create(**payload)
        return resp.choices[0].message.content


class GenAIClient(LLMClient):
    def __init__(
        self,
        model="gemini-2.5-flash",
        api_key=None,
        temperature=None,
        reasoning=None,
    ):

        self.client = genai.Client(api_key=api_key or os.environ["GENAI_API_KEY"])
        self.model = model
        self.temperature = temperature
        self.reasoning = reasoning

    def chat(self, messages):
        

        system = ""
        user_parts = []

        for m in messages:
            if m["role"] == "system":
                system += m["content"] + "\n"
            elif m["role"] == "user":
                user_parts.append(m["content"])
            else:
                user_parts.append(f'{m["role"].upper()}: {m["content"]}')

        config_kwargs = {}
        if system.strip():
            config_kwargs["system_instruction"] = system.strip()
        if self.temperature is not None:
            config_kwargs["temperature"] = self.temperature

        # This may vary by SDK/model support
        if self.reasoning is not None:
            budget_map = {
                "none": 0,
                "low": 1024,
                "medium": 4096,
                "high": 8192,
            }
            config_kwargs["thinking_config"] = genai.types.ThinkingConfig(
                thinking_budget=budget_map.get(self.reasoning, 4096)
            )

        chat = self.client.chats.create(
            model=self.model,
            config=genai.types.GenerateContentConfig(**config_kwargs),
        )
        resp = chat.send_message("\n\n".join(user_parts))
        return resp.text


class UfalClient(LLMClient):
    def __init__(
        self,
        model="Apertus-70B-8b_4x3090.RedHatAI/Apertus-70B-Instruct-2509-FP8-dynamic",
        api_token=None,
        temperature=0.0,
        reasoning=None,
    ):
        self.model = model
        self.api_url = "https://ai.ufal.mff.cuni.cz/api/chat/completions"
        self.timeout = 120
        self.api_token = api_token or os.environ["UFAL_API_KEY"]
        self.temperature = temperature
        self.reasoning = reasoning

    def chat(self, messages):
        payload = {
            "model": self.model,
            "messages": messages,
        }

        if self.temperature is not None:
            payload["temperature"] = self.temperature

        if self.reasoning is not None:
            payload["reasoning"] = {"effort": self.reasoning}

        r = requests.post(
            self.api_url,
            headers={
                "Authorization": f"Bearer {self.api_token}",
                "Content-Type": "application/json",
            },
            data=json.dumps(payload),
            timeout=self.timeout,
        )
        r.raise_for_status()
        data = r.json()

        try:
            return data["choices"][0]["message"]["content"]
        except (KeyError, IndexError, TypeError):
            raise RuntimeError(f"Unexpected UFAL response shape: {data}")


def get_llm(
    provider: str,
    model: str = None,
    temperature: float = None,
    reasoning: str = None,
) -> LLMClient:
    kwargs = {
        "temperature": temperature,
        "reasoning": reasoning,
    }
    if model is not None:
        kwargs["model"] = model

    if provider == "openai":
        return OpenAIClient(**kwargs)

    elif provider == "genai":
        return GenAIClient(**kwargs)

    elif provider == "ufalai":
        return UfalClient(**kwargs)

    elif provider == "groq":
        return GroqClient(**kwargs)

    else:
        raise ValueError(f"Unknown provider: {provider}")

In [26]:
import json 
def get_json(string):
    stripped = ""
    braces = 0
    for s in string:
        if s == "{":
            braces += 1
        if braces >= 1:
            stripped += s
        if s == "}":
            braces -= 1
    return json.loads(stripped, strict=False)


In [27]:
def retry(func, *args, retries=1, delay=30, **kwargs):
    """
    Call func with retries and delay on Exception.
    """
    for attempt in range(retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if attempt == retries:
                # Last attempt failed, re-raise
                raise
            print(f"Error: {e}. {func} Another attempt...")
            time.sleep(delay)

In [53]:
import csv
import sys
from typing import Dict, Any
import time

import numpy as np
import pandas as pd




def induce_new_attributes(
    llm: LLMClient,
    residuals: Dict[Any, str],
    attribute_list: Dict[str, str],
) -> Dict[str, str]:
    system_prompt = f"""
These are the residual texts from court verdicts after extracting the following attributes:
{attribute_list}

What other pieces of information do you see repeatedly in the residual texts?
Return a JSON with the emergent attribute names as keys and descriptions as values.
The attributes should be as granular as possible, but do not repeat the attributes that were already defined.
""".strip()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": str(residuals)},
    ]

    response = llm.chat(messages)
    result = get_json(response)
    print("Candidate attributes:", result)
    return result


def populate_new_attributes(
    llm: LLMClient,
    case_id: Any,
    sentence: str,
    attribute_list: Dict[str, str],
    attributes_structured: Dict[Any, Dict[str, Any]],
    residuals: Dict[Any, str],
) -> None:
    system_prompt = f"""
You extract values for attributes from text.
Return only information from the text, never make anything up.
If the value for an attribute is not provided, return value "n/a".
The date format is DD.MM.YYYY.
The time format is HH:MM.
Decimal numbers are rounded to two digits after the decimal point: #.##.
Format your output as JSON.
The values must be in the original language - Slovak.

Extract the following attributes:
{attribute_list}
""".strip()

    extraction_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sentence},
    ]
    extraction_response = llm.chat(extraction_messages)
    new_attributes = get_json(extraction_response)

    attributes_structured[case_id].update(new_attributes)

    residual_instruction = """
What other information was there in the statement but is not covered by the attributes?
Write one sentence containing the extra information.
Output a JSON with one attribute: "residual" where the value is the sentence with extra information.
Only output information from the original text.
The sentence must be in the original language - Slovak.
If there is no extra information, output {"residual": "n/a"}.
""".strip()

    residual_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sentence},
        {"role": "assistant", "content": str(new_attributes)},
        {"role": "user", "content": residual_instruction},
    ]
    residual_response = llm.chat(residual_messages)
    residual_json = get_json(residual_response)

    residual_value = residual_json.get("residual", "n/a")
    attributes_structured[case_id]["residual"] = residual_value
    residuals[case_id] = residual_value

    print(f"Processed case {case_id}:")
    print({**new_attributes, "residual": residual_value})


def check_population(
    attributes_structured: Dict[Any, Dict[str, Any]],
    candidate_attributes: Dict[str, str],
    threshold: float = 0.85,
) -> Dict[str, str]:
    if not candidate_attributes:
        return {}

    df = pd.DataFrame.from_dict(attributes_structured, orient="index").reset_index()
    df.replace("n/a", np.nan, inplace=True)

    missing_ratio = df.isna().sum() / len(df)
    final_attributes = {}

    for attribute, description in candidate_attributes.items():
        if attribute not in df.columns:
            print(f"Dropping attribute: {attribute} (not present in extracted data)")
            continue

        if missing_ratio[attribute] <= threshold:
            final_attributes[attribute] = description
            print(f"New attribute accepted: {attribute}")
        else:
            print(f"Dropping attribute: {attribute}")

    return final_attributes


def run(
    df: pd.DataFrame,
    llm: LLMClient,
    initial_attributes: Dict[str, str],
    batch_size: int,
) -> pd.DataFrame:
    attributes_structured: Dict[Any, Dict[str, Any]] = {}
    residuals: Dict[Any, str] = {}
    attribute_list = dict(initial_attributes)

    for i, (_, row) in enumerate(df.iterrows()):
        case_id = row["judgement_slt_id"]
        sentence = row["judgement_factual_sentence"]

        attributes_structured[case_id] = {"sentence": sentence}

        print(f"\nProcessing row {i}")
        retry(
            populate_new_attributes,
            llm,
            case_id,
            sentence,
            attribute_list,
            attributes_structured,
            residuals,
        )

        if i > 0 and i % batch_size == 0:
            candidate_attributes = retry(
                induce_new_attributes,
                llm,
                residuals,
                attribute_list,
            )

            for existing_case_id, case_data in attributes_structured.items():
                residual = case_data.get("residual", "n/a")
                if residual != "n/a":
                    retry(
                        populate_new_attributes,
                        llm,
                        existing_case_id,
                        residual,
                        candidate_attributes,
                        attributes_structured,
                        residuals,
                    )

            new_attributes = check_population(attributes_structured, candidate_attributes)
            attribute_list.update(new_attributes)

    return pd.DataFrame.from_dict(attributes_structured, orient="index").reset_index(), attribute_list



In [48]:
import pandas as pd
import csv
import sys
csv.field_size_limit(sys.maxsize)

df = pd.read_csv(
    "../data/categorization/DKK-dev-final.csv",
    sep=",",
    quoting=1,  # csv.QUOTE_ALL
    quotechar='"',
    escapechar="\\",
    engine="python"  # Required when dealing with multiline quoted fields
)

# Check the first few rows
marenky = df[df["judgement_model_short"] == "A"]
pod_vlivem = df[df["judgement_model_short"] == "B"]
s_nasledkem = df[df["judgement_model_short"] == "C"]
jina = df[df["judgement_model_short"] == "D"]
nedopravni = df[df["judgement_model_short"] == "E"]

In [55]:
BATCH_SIZE = 10

llm = get_llm(
    provider="openai",
    model = "gpt-5-mini",
    temperature=1.0,
    reasoning="minimal",
)

initial_attributes = {
    "dateOfOffense": "Date when the offense occurred",
}

# Trying out on first 20 statements
sample = pod_vlivem[:20]

df_attributes, attribute_list = run(
    df=sample,
    llm=llm,
    initial_attributes=initial_attributes,
    batch_size=BATCH_SIZE,

    
)

df_attributes.to_csv("pod_vlivem_attributes.csv", index=False)

print(df_attributes.head())





Processing row 0
Processed case 2276619:
{'dateOfOffense': '05.02.2018', 'residual': 'V čase o 21.08 hod. viedol osobné motorové vozidlo zn. Citroën Saxo, ev. č. M. XXXDD po ceste II/583 po ulici sv. Martina v obci Terchová v smere jazdy od obce Terchová Vyšné Kamence na časť obce Terchová - biely Potok, ovplyvnený alkoholom vo výške 0,84 mg etanolu na liter vydychovaného vzduchu (čo predstavuje 1,75 promile), pričom porušil ustanovenia § 4 ods. 2 písm. d) Zákona č. 8/2009 Z.z. o cestnej premávke.'}

Processing row 1
Processed case 1808110:
{'dateOfOffense': '01.01.2018', 'residual': 'V texte sa uvádza, že obvinený viedol osobné motorové vozidlo zn. U. XXX, evidenčného čísla XSX 4913, bol zastavený a kontrolovaný policajnou hliadkou Obvodného oddelenia PZ A. v obci A. po ceste I/51, podrobený dychovej skúške meračom J. 6020 plus č. A409625, pričom boli namerané hodnoty alkoholu v dychu 0,64 mg/l (1,33 ‰) o 23:41 hod. a 0,60 mg/l (1,25 ‰) pri opakovanej skúške o 23:59 hod.'}

Processin